In [1]:
from KUCBVI_new import *

In [2]:
import os, json
import numpy as np
import matplotlib.pyplot as plt
from contextlib import redirect_stdout

def _pad_to_max_length(curves, pad_value=np.nan):
    max_len = max(len(c) for c in curves)
    out = np.full((len(curves), max_len), pad_value, dtype=float)
    for i, c in enumerate(curves):
        out[i, :len(c)] = c
    return out

def run_5_seeds_model_based(
    train_fn,
    train_kwargs,
    seeds=(0,1,2,3,4),
    out_dir="plot2_model_based",
    label="model_based",
    silence_prints=True,
):
    """
    Runner for model-based training loop that returns:
      policy, reward_model, avg_true_rewards, avg_est_rewards, W, steps, queries

    Stores ONLY:
      - avg_true_rewards curves
      - mean/std across seeds
      - plot
      - summary.json
    """
    os.makedirs(out_dir, exist_ok=True)

    per_seed_curves = []
    final_returns = []
    total_steps = []
    total_queries = []

    for seed in seeds:
        kwargs = dict(train_kwargs)
        kwargs["seed"] = seed

        log_path = os.path.join(out_dir, f"{label}_seed{seed}_log.txt")
        print(f"\n=== Running seed {seed} ===")

        if silence_prints:
            with open(log_path, "w") as f, redirect_stdout(f):
                policy, reward_model, avg_true_rewards, avg_est_rewards, W, steps, queries = train_fn(**kwargs)
        else:
            policy, reward_model, avg_true_rewards, avg_est_rewards, W, steps, queries = train_fn(**kwargs)

        curve = np.asarray(avg_true_rewards, dtype=float)
        per_seed_curves.append(curve)

        final_returns.append(float(curve[-1]) if len(curve) else np.nan)
        total_steps.append(int(steps))
        total_queries.append(int(queries))

        np.save(os.path.join(out_dir, f"{label}_seed{seed}_curve.npy"), curve)

    curves_mat = _pad_to_max_length(per_seed_curves, pad_value=np.nan)
    mean_curve = np.nanmean(curves_mat, axis=0)
    std_curve  = np.nanstd(curves_mat, axis=0)

    x_updates = np.arange(len(mean_curve), dtype=float)

    np.save(os.path.join(out_dir, f"{label}_mean_curve.npy"), mean_curve)
    np.save(os.path.join(out_dir, f"{label}_std_curve.npy"), std_curve)
    np.save(os.path.join(out_dir, f"{label}_x_updates.npy"), x_updates)

    summary = {
        "label": label,
        "seeds": list(seeds),
        "train_kwargs": train_kwargs,
        "final_return_per_seed": final_returns,
        "final_return_mean": float(np.nanmean(final_returns)),
        "final_return_std": float(np.nanstd(final_returns)),
        "total_steps_per_seed": total_steps,
        "total_queries_per_seed": total_queries,
        "curve_len_per_seed": [int(len(c)) for c in per_seed_curves],
    }

    with open(os.path.join(out_dir, f"{label}_summary.json"), "w") as f:
        json.dump(summary, f, indent=2)

    # Plot (x-axis = iteration index n)
    plt.figure(figsize=(8, 4.8))
    for c in per_seed_curves:
        plt.plot(np.arange(len(c)), c, alpha=0.25, linewidth=1)

    plt.plot(x_updates, mean_curve, linewidth=2, label="mean (5 seeds)")
    plt.fill_between(
        x_updates,
        mean_curve - std_curve,
        mean_curve + std_curve,
        alpha=0.2,
        label="±1 std",
    )

    plt.xlabel("Training iteration (outer loop n)")
    plt.ylabel("Avg true return")
    plt.title(label)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()

    fig_path = os.path.join(out_dir, f"{label}.png")
    plt.savefig(fig_path, dpi=200)
    plt.close()

    print("\nSaved:", fig_path)
    print(
        "Final avg true return over seeds:",
        f"{summary['final_return_mean']:.4f} ± {summary['final_return_std']:.4f}",
    )

    return summary


In [3]:
noise_list = [0.1, 0.5, 0.8]

base_kwargs = dict(
    N=200,          # ≈ 15k PG steps total
    m=50,
    k=6,
    eta=0.1,
    epsilon=0.1,
    grid_size=8,
    danger=[7,1],
    goal=[4,5],
    wall=[2,5],
    horizon=50,
    coins=None,
)

for sparse in [False, True]:
    for noise in noise_list:
        kwargs = dict(base_kwargs)
        kwargs["noise"] = noise
        kwargs["sparse"] = sparse

        label = f"kucbvi_noise{noise}_sparse{sparse}"
        run_5_seeds_model_based(
            train_fn=train,
            train_kwargs=kwargs,
            seeds=(0, 2, 4),
            out_dir="plot2_model_based_kucbvi",
            label=label,
            silence_prints=0,
        )




=== Running seed 0 ===
hi
0
Iter 00: avg_label=0.380, avg_true_reward=4.030,coins_this_episode=1.00
1
Iter 01: avg_label=0.620, avg_true_reward=5.419,coins_this_episode=1.00
2
Iter 02: avg_label=0.500, avg_true_reward=4.882,coins_this_episode=1.00
3
Iter 03: avg_label=0.660, avg_true_reward=5.693,coins_this_episode=0.00
4
Iter 04: avg_label=0.580, avg_true_reward=5.396,coins_this_episode=1.00
5
Iter 05: avg_label=0.440, avg_true_reward=4.342,coins_this_episode=1.00
6
Iter 06: avg_label=0.580, avg_true_reward=5.473,coins_this_episode=2.00
7
Iter 07: avg_label=0.680, avg_true_reward=5.975,coins_this_episode=1.00
8
Iter 08: avg_label=0.780, avg_true_reward=6.456,coins_this_episode=0.00
9
Iter 09: avg_label=0.780, avg_true_reward=6.722,coins_this_episode=1.00
10
Iter 10: avg_label=1.120, avg_true_reward=8.650,coins_this_episode=2.00
11
Iter 11: avg_label=0.780, avg_true_reward=6.562,coins_this_episode=1.00
12
Iter 12: avg_label=0.760, avg_true_reward=6.301,coins_this_episode=0.00
13
Iter 

Iter 108: avg_label=4.540, avg_true_reward=28.522,coins_this_episode=3.00
109
Iter 109: avg_label=4.600, avg_true_reward=29.038,coins_this_episode=3.00
110
Iter 110: avg_label=4.480, avg_true_reward=28.644,coins_this_episode=3.00
111
Iter 111: avg_label=4.540, avg_true_reward=28.742,coins_this_episode=3.00
112
Iter 112: avg_label=4.680, avg_true_reward=29.432,coins_this_episode=3.00
113
Iter 113: avg_label=4.500, avg_true_reward=28.646,coins_this_episode=3.00
114
Iter 114: avg_label=4.640, avg_true_reward=29.636,coins_this_episode=3.00
115
Iter 115: avg_label=4.520, avg_true_reward=28.543,coins_this_episode=3.00
116
Iter 116: avg_label=4.620, avg_true_reward=29.103,coins_this_episode=3.00
117
Iter 117: avg_label=4.860, avg_true_reward=31.009,coins_this_episode=3.00
118
Iter 118: avg_label=4.680, avg_true_reward=29.645,coins_this_episode=3.00
119
Iter 119: avg_label=4.540, avg_true_reward=28.836,coins_this_episode=2.00
120
Iter 120: avg_label=4.700, avg_true_reward=29.860,coins_this_epi

Iter 14: avg_label=1.000, avg_true_reward=7.713,coins_this_episode=0.00
15
Iter 15: avg_label=0.940, avg_true_reward=7.327,coins_this_episode=0.00
16
Iter 16: avg_label=0.960, avg_true_reward=7.845,coins_this_episode=1.00
17
Iter 17: avg_label=1.160, avg_true_reward=8.533,coins_this_episode=1.00
18
Iter 18: avg_label=1.160, avg_true_reward=8.723,coins_this_episode=2.00
19
Iter 19: avg_label=1.220, avg_true_reward=8.978,coins_this_episode=1.00
20
Iter 20: avg_label=1.260, avg_true_reward=9.172,coins_this_episode=1.00
21
Iter 21: avg_label=1.180, avg_true_reward=8.856,coins_this_episode=0.00
22
Iter 22: avg_label=1.280, avg_true_reward=9.577,coins_this_episode=1.00
23
Iter 23: avg_label=1.240, avg_true_reward=9.171,coins_this_episode=1.00
24
Iter 24: avg_label=1.360, avg_true_reward=9.338,coins_this_episode=0.00
25
Iter 25: avg_label=1.360, avg_true_reward=10.092,coins_this_episode=1.00
26
Iter 26: avg_label=1.420, avg_true_reward=10.071,coins_this_episode=1.00
27
Iter 27: avg_label=1.48

Iter 122: avg_label=2.920, avg_true_reward=20.234,coins_this_episode=2.00
123
Iter 123: avg_label=2.960, avg_true_reward=20.386,coins_this_episode=2.00
124
Iter 124: avg_label=2.880, avg_true_reward=19.704,coins_this_episode=2.00
125
Iter 125: avg_label=2.940, avg_true_reward=19.866,coins_this_episode=2.00
126
Iter 126: avg_label=2.920, avg_true_reward=20.094,coins_this_episode=2.00
127
Iter 127: avg_label=2.980, avg_true_reward=20.038,coins_this_episode=2.00
128
Iter 128: avg_label=3.000, avg_true_reward=20.736,coins_this_episode=2.00
129
Iter 129: avg_label=2.920, avg_true_reward=20.257,coins_this_episode=2.00
130
Iter 130: avg_label=2.920, avg_true_reward=19.801,coins_this_episode=2.00
131
Iter 131: avg_label=2.980, avg_true_reward=20.523,coins_this_episode=2.00
132
Iter 132: avg_label=2.960, avg_true_reward=20.415,coins_this_episode=2.00
133
Iter 133: avg_label=2.940, avg_true_reward=20.342,coins_this_episode=2.00
134
Iter 134: avg_label=2.940, avg_true_reward=20.022,coins_this_epi

Iter 28: avg_label=1.740, avg_true_reward=11.996,coins_this_episode=1.00
29
Iter 29: avg_label=1.360, avg_true_reward=9.940,coins_this_episode=2.00
30
Iter 30: avg_label=1.500, avg_true_reward=10.536,coins_this_episode=0.00
31
Iter 31: avg_label=1.660, avg_true_reward=11.196,coins_this_episode=1.00
32
Iter 32: avg_label=1.640, avg_true_reward=11.089,coins_this_episode=1.00
33
Iter 33: avg_label=1.580, avg_true_reward=10.757,coins_this_episode=1.00
34
Iter 34: avg_label=1.720, avg_true_reward=11.800,coins_this_episode=1.00
35
Iter 35: avg_label=1.640, avg_true_reward=11.269,coins_this_episode=1.00
36
Iter 36: avg_label=1.840, avg_true_reward=12.339,coins_this_episode=1.00
37
Iter 37: avg_label=1.820, avg_true_reward=12.655,coins_this_episode=2.00
38
Iter 38: avg_label=2.120, avg_true_reward=14.079,coins_this_episode=1.00
39
Iter 39: avg_label=1.900, avg_true_reward=13.031,coins_this_episode=1.00
40
Iter 40: avg_label=2.240, avg_true_reward=14.926,coins_this_episode=3.00
41
Iter 41: avg_

...

Iter 79: avg_label=0.340, avg_true_reward=0.907,coins_this_episode=0.00
80
Iter 80: avg_label=0.200, avg_true_reward=0.615,coins_this_episode=1.00
81
Iter 81: avg_label=0.180, avg_true_reward=0.526,coins_this_episode=0.00
82
Iter 82: avg_label=0.200, avg_true_reward=0.570,coins_this_episode=0.00
83
Iter 83: avg_label=0.200, avg_true_reward=0.570,coins_this_episode=0.00
84
Iter 84: avg_label=0.200, avg_true_reward=0.602,coins_this_episode=0.00
85
Iter 85: avg_label=0.220, avg_true_reward=0.627,coins_this_episode=0.00
86
Iter 86: avg_label=0.200, avg_true_reward=0.582,coins_this_episode=0.00
87
Iter 87: avg_label=0.240, avg_true_reward=0.684,coins_this_episode=1.00
88
Iter 88: avg_label=0.220, avg_true_reward=0.608,coins_this_episode=0.00
89
Iter 89: avg_label=0.160, avg_true_reward=0.534,coins_this_episode=0.00
90
Iter 90: avg_label=0.140, avg_true_reward=0.453,coins_this_episode=0.00
91
Iter 91: avg_label=0.240, avg_true_reward=0.652,coins_this_episode=0.00
92
Iter 92: avg_label=0.160,

Iter 186: avg_label=0.000, avg_true_reward=0.025,coins_this_episode=0.00
187
Iter 187: avg_label=0.000, avg_true_reward=0.025,coins_this_episode=0.00
188
Iter 188: avg_label=0.000, avg_true_reward=0.128,coins_this_episode=0.00
189
Iter 189: avg_label=0.000, avg_true_reward=0.125,coins_this_episode=0.00
190
Iter 190: avg_label=0.000, avg_true_reward=0.071,coins_this_episode=0.00
191
Iter 191: avg_label=0.000, avg_true_reward=0.051,coins_this_episode=0.00
192
Iter 192: avg_label=0.000, avg_true_reward=0.048,coins_this_episode=0.00
193
Iter 193: avg_label=0.000, avg_true_reward=0.028,coins_this_episode=0.00
194
Iter 194: avg_label=0.000, avg_true_reward=0.063,coins_this_episode=0.00
195
Iter 195: avg_label=0.020, avg_true_reward=0.137,coins_this_episode=0.00
196
Iter 196: avg_label=0.000, avg_true_reward=0.043,coins_this_episode=0.00
197
Iter 197: avg_label=0.000, avg_true_reward=0.088,coins_this_episode=0.00
198
Iter 198: avg_label=0.060, avg_true_reward=0.243,coins_this_episode=0.00
199

Iter 95: avg_label=0.520, avg_true_reward=1.301,coins_this_episode=0.00
96
Iter 96: avg_label=0.500, avg_true_reward=1.242,coins_this_episode=0.00
97
Iter 97: avg_label=0.500, avg_true_reward=1.289,coins_this_episode=0.00
98
Iter 98: avg_label=0.460, avg_true_reward=1.170,coins_this_episode=0.00
99
Iter 99: avg_label=0.360, avg_true_reward=0.934,coins_this_episode=0.00
100
Iter 100: avg_label=0.460, avg_true_reward=1.138,coins_this_episode=0.00
101
Iter 101: avg_label=0.280, avg_true_reward=0.762,coins_this_episode=0.00
102
Iter 102: avg_label=0.500, avg_true_reward=1.222,coins_this_episode=0.00
103
Iter 103: avg_label=0.540, avg_true_reward=1.331,coins_this_episode=0.00
104
Iter 104: avg_label=0.560, avg_true_reward=1.392,coins_this_episode=1.00
105
Iter 105: avg_label=0.640, avg_true_reward=1.635,coins_this_episode=0.00
106
Iter 106: avg_label=0.600, avg_true_reward=1.497,coins_this_episode=0.00
107
Iter 107: avg_label=0.740, avg_true_reward=1.803,coins_this_episode=0.00
108
Iter 108

Iter 02: avg_label=0.560, avg_true_reward=1.848,coins_this_episode=1.00
3
Iter 03: avg_label=0.440, avg_true_reward=1.605,coins_this_episode=1.00
4
Iter 04: avg_label=0.460, avg_true_reward=1.728,coins_this_episode=1.00
5
Iter 05: avg_label=0.600, avg_true_reward=1.944,coins_this_episode=2.00
6
Iter 06: avg_label=0.480, avg_true_reward=1.697,coins_this_episode=2.00
7
Iter 07: avg_label=0.440, avg_true_reward=1.734,coins_this_episode=2.00
8
Iter 08: avg_label=0.540, avg_true_reward=1.824,coins_this_episode=1.00
9
Iter 09: avg_label=0.400, avg_true_reward=1.686,coins_this_episode=1.00
10
Iter 10: avg_label=0.600, avg_true_reward=1.998,coins_this_episode=1.00
11
Iter 11: avg_label=0.360, avg_true_reward=1.442,coins_this_episode=1.00
12
Iter 12: avg_label=0.500, avg_true_reward=1.676,coins_this_episode=1.00
13
Iter 13: avg_label=0.640, avg_true_reward=1.962,coins_this_episode=0.00
14
Iter 14: avg_label=0.540, avg_true_reward=1.959,coins_this_episode=2.00
15
Iter 15: avg_label=0.620, avg_tr

Iter 112: avg_label=1.140, avg_true_reward=4.126,coins_this_episode=2.00
113
Iter 113: avg_label=1.240, avg_true_reward=4.333,coins_this_episode=3.00
114
Iter 114: avg_label=1.060, avg_true_reward=3.946,coins_this_episode=1.00
115
Iter 115: avg_label=1.100, avg_true_reward=4.106,coins_this_episode=2.00
116
Iter 116: avg_label=1.320, avg_true_reward=4.480,coins_this_episode=2.00
117
Iter 117: avg_label=1.100, avg_true_reward=4.073,coins_this_episode=2.00
118
Iter 118: avg_label=1.060, avg_true_reward=4.046,coins_this_episode=2.00
119
Iter 119: avg_label=1.120, avg_true_reward=4.112,coins_this_episode=2.00
120
Iter 120: avg_label=1.180, avg_true_reward=4.204,coins_this_episode=2.00
121
Iter 121: avg_label=1.020, avg_true_reward=3.959,coins_this_episode=2.00
122
Iter 122: avg_label=1.000, avg_true_reward=3.939,coins_this_episode=2.00
123
Iter 123: avg_label=1.120, avg_true_reward=4.126,coins_this_episode=2.00
124
Iter 124: avg_label=1.020, avg_true_reward=3.893,coins_this_episode=2.00
125